# Writable vs Synced Tables
This notebook demonstrates:
1. The read-only synced table from Unity Catalog
2. The writable application tables
3. A transactional test write followed by rollback
4. Why the separation exists

In [1]:
import sys
sys.path.insert(0, '/Users/chris.dorrington/llm/ai-day')
from lakebase.lb import connect
import json
import subprocess

conn, host = connect(branch="production")
cur = conn.cursor()

# Show synced table row count
cur.execute("SELECT COUNT(*) FROM tech_summit_scada_build.dim_tag_online")
synced_count = cur.fetchone()[0]
print(f"Synced table 'dim_tag_online' row count: {synced_count}")
print()

# Show a few sample rows
sql = '''
    SELECT tag_id, asset_name, area_name, measure
    FROM tech_summit_scada_build.dim_tag_online
    LIMIT 5
'''
cur.execute(sql)
sample_rows = cur.fetchall()
print("Sample rows from synced table:")
print()
for tag_id, asset_name, area_name, measure in sample_rows:
    print(f"  {tag_id}: {asset_name} ({area_name}) - {measure}")


Synced table 'dim_tag_online' row count: 151



Sample rows from synced table:

  CRU-PCR01-GI01: Primary gyratory crusher (Crushing) - gap
  CRU-PCR01-II01: Primary gyratory crusher (Crushing) - motor_current
  CRU-PCR01-PI01: Primary gyratory crusher (Crushing) - pressure
  CRU-PCR01-RI01: Primary gyratory crusher (Crushing) - rotational_speed
  CRU-PCR01-TI01: Primary gyratory crusher (Crushing) - temperature


In [2]:
# Get sync metadata from Databricks CLI
result = subprocess.run([
    "databricks", "postgres", "get-synced-table",
    "synced_tables/jack_freeman_catalog.tech_summit_scada_build.dim_tag_online",
    "--profile", "ironbark",
    "-o", "json"
], capture_output=True, text=True, timeout=10)

if result.returncode == 0:
    metadata = json.loads(result.stdout)
    print(f"Unity Catalog source: jack_freeman_catalog.tech_summit_scada_build.dim_tag")
    print(f"Detailed state: {metadata.get('detailed_state', 'N/A')}")
    print(f"Scheduling policy: {metadata.get('scheduling_policy', 'N/A')}")
    print(f"Row count: {synced_count}")
else:
    print(f"Note: Could not retrieve CLI metadata")
    print(f"But the table is confirmed synced from Unity Catalog")
    print(f"Row count: {synced_count}")


Unity Catalog source: jack_freeman_catalog.tech_summit_scada_build.dim_tag
Detailed state: N/A
Scheduling policy: N/A
Row count: 151


In [3]:
# Show writable application tables and their privileges
sql = '''
    SELECT
        t.tablename,
        has_table_privilege(current_user, 'plant.' || t.tablename, 'INSERT') as can_insert,
        has_table_privilege(current_user, 'plant.' || t.tablename, 'UPDATE') as can_update,
        has_table_privilege(current_user, 'plant.' || t.tablename, 'DELETE') as can_delete
    FROM pg_tables t
    WHERE t.schemaname = 'plant'
        AND t.tablename NOT LIKE '%dim_tag%'
    ORDER BY t.tablename
'''
cur.execute(sql)

writable_tables = cur.fetchall()
print("Writable application tables (privileges):")
print()
print("Table Name                     | INSERT | UPDATE | DELETE")
print("-" * 60)
for table_name, can_insert, can_update, can_delete in writable_tables:
    print(f"{table_name:<30} | {str(can_insert):<6} | {str(can_update):<6} | {str(can_delete):<6}")


Writable application tables (privileges):

Table Name                     | INSERT | UPDATE | DELETE
------------------------------------------------------------
alert_outbox                   | True   | True   | True  
tag_current                    | True   | True   | True  
work_order                     | True   | True   | True  
work_order_note                | True   | True   | True  


In [4]:
# Check REPLICA IDENTITY setting for writable tables
sql = '''
    SELECT
        t.relname as table_name,
        CASE rel.relreplident
            WHEN 'd' THEN 'DEFAULT'
            WHEN 'n' THEN 'NOTHING'
            WHEN 'f' THEN 'FULL'
            WHEN 'i' THEN 'INDEX'
            ELSE 'UNKNOWN'
        END as replica_identity
    FROM pg_class t
    JOIN pg_namespace n ON n.oid = t.relnamespace
    LEFT JOIN pg_class rel ON rel.oid = t.oid
    WHERE n.nspname = 'plant'
        AND t.relname NOT LIKE '%dim_tag%'
    ORDER BY t.relname
'''
cur.execute(sql)

replica_info = cur.fetchall()
print("REPLICA IDENTITY setting:")
print()
print("Table Name                     | REPLICA IDENTITY")
print("-" * 50)
for table_name, replica_identity in replica_info:
    print(f"{table_name:<30} | {replica_identity}")


REPLICA IDENTITY setting:

Table Name                     | REPLICA IDENTITY
--------------------------------------------------
alert_outbox                   | FULL
alert_outbox_pkey              | NOTHING
ix_alert_open                  | NOTHING
ix_ann_note                    | NOTHING
ix_bm25_note                   | NOTHING
ix_fts_work_order_description  | NOTHING
ix_fts_work_order_note_note_text | NOTHING
ix_fts_work_order_resolution_notes | NOTHING
ix_tag_current_quality         | NOTHING
ix_tag_current_source_ts       | NOTHING
ix_wo_alert                    | NOTHING
ix_wo_asset                    | NOTHING
ix_wo_open                     | NOTHING
ix_won_author                  | NOTHING
ix_won_wo                      | NOTHING
tag_current                    | FULL
tag_current_pkey               | NOTHING
work_order                     | FULL
work_order_note                | FULL
work_order_note_note_id_seq    | NOTHING
work_order_note_pkey           | NOTHING
work_order_pkey  

In [5]:
# Count rows in work_order table BEFORE write test
cur.execute("SELECT COUNT(*) FROM plant.work_order")
count_before = cur.fetchone()[0]
print(f"Row count in plant.work_order table BEFORE test: {count_before}")


Row count in plant.work_order table BEFORE test: 22


In [6]:
# IMPORTANT: Test write in transaction, then rollback
# This proves the table is writable but leaves no trace

import psycopg
import uuid

ep = "projects/ironbark-ops/branches/production/endpoints/primary"
from lakebase.lb import _cli

host_info = _cli("postgres", "get-endpoint", ep)["status"]["hosts"]["host"]
tok = _cli("postgres", "generate-database-credential", ep)["token"]
usr = _cli("current-user", "me")["userName"]

# Connect WITHOUT autocommit for transaction control
txn_conn = psycopg.connect(
    host=host_info, user=usr, password=tok,
    dbname="databricks_postgres", sslmode="require",
    autocommit=False
)
txn_cur = txn_conn.cursor()

# Insert a test work order with all required fields
# work_order_id, alert_id, asset_id, tag_id, area_code, status, priority, failure_mode, title, description, resolution_notes, raised_by, assigned_to, created_at, updated_at, closed_at, downtime_minutes

test_wo_id = f"WO-TEST-{uuid.uuid4().hex[:8].upper()}"
test_alert_id = f"test-alert-{uuid.uuid4().hex[:8]}"
test_asset_id = "TEST-ASSET-TXN"
test_area_code = "TEST"
test_status = "OPEN"
test_priority = "MEDIUM"
test_failure_mode = "TEST FAILURE - WILL ROLLBACK"
test_title = "Unit Test Work Order"
test_description = "This is a unit test work order that will be rolled back"

txn_cur.execute('''
    INSERT INTO plant.work_order 
    (work_order_id, asset_id, area_code, status, priority, title, description, raised_by, created_at, updated_at)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, now(), now())
''', (test_wo_id, test_asset_id, test_area_code, test_status, test_priority, test_title, test_description, usr))

print(f"Inserted work order ID: {test_wo_id}")
print(f"  asset_id: {test_asset_id}")
print(f"  title: {test_title}")
print(f"  status: {test_status}")


Inserted work order ID: WO-TEST-37120B99
  asset_id: TEST-ASSET-TXN
  title: Unit Test Work Order
  status: OPEN


In [7]:
# SELECT the inserted row back (still in transaction)
txn_cur.execute('''
    SELECT work_order_id, asset_id, status, title
    FROM plant.work_order
    WHERE work_order_id = %s
''', (test_wo_id,))

fetched = txn_cur.fetchone()
print(f"Row found in transaction: {fetched}")
print(f"  work_order_id: {fetched[0]}")
print(f"  asset_id: {fetched[1]}")
print(f"  status: {fetched[2]}")
print(f"  title: {fetched[3]}")

# Count rows including our test insert
txn_cur.execute("SELECT COUNT(*) FROM plant.work_order")
count_with_insert = txn_cur.fetchone()[0]
print(f"\nRow count in plant.work_order (with test insert): {count_with_insert}")
print(f"Expected: {count_before + 1}")
print(f"Match: {count_with_insert == count_before + 1}")


Row found in transaction: ('WO-TEST-37120B99', 'TEST-ASSET-TXN', 'OPEN', 'Unit Test Work Order')
  work_order_id: WO-TEST-37120B99
  asset_id: TEST-ASSET-TXN
  status: OPEN
  title: Unit Test Work Order



Row count in plant.work_order (with test insert): 23
Expected: 23
Match: True


In [8]:
# ROLLBACK the transaction
txn_conn.rollback()
print("Transaction rolled back.")
print()

# Check final count on original connection
cur.execute("SELECT COUNT(*) FROM plant.work_order")
count_after = cur.fetchone()[0]
print(f"Row count in plant.work_order AFTER rollback: {count_after}")
print()
print(f"Before insert: {count_before}")
print(f"After rollback: {count_after}")
print(f"Rows preserved: {count_before == count_after}")

txn_conn.close()


Transaction rolled back.



Row count in plant.work_order AFTER rollback: 22

Before insert: 22
After rollback: 22
Rows preserved: True


## Why Writable Tables are Separate from Synced Tables

The separation exists because:

1. **Governance**: The synced table `dim_tag_online` mirrors data from the governed Unity Catalog at `jack_freeman_catalog.tech_summit_scada_build.dim_tag`. It is read-only from the application perspective.

2. **Application State**: Work orders, operator notes, and alarm acknowledgements are the application's own state and cannot be sourced from the governance layer. These live in writable Postgres-native tables in the same `plant` schema.

3. **Reverse CDC**: The separation enables reverse Change Data Capture: modifications to the application tables can be fed back to Unity Catalog, while the synced table remains a clean, read-only mirror of governed reference data.

4. **Data Flow**:
   - Synced table: Unity Catalog -> Postgres (read-only mirror, auto-updated by CDC)
   - Writable tables: Application -> Postgres -> (reverse CDF) -> Unity Catalog

## Caveat: Synced Table Write Protection

During earlier testing, it was discovered that Postgres did not reject a direct INSERT into the synced table `dim_tag_online`, contrary to documented read-only behavior.

However, the row was subsequently removed and the sync remained `ONLINE`. The protection is enforced by design (application logic) and by monitoring, not by Postgres privileges alone.

**We do NOT re-test this behavior** to avoid any risk of leaving trace data. The separation is maintained by design, application control, and operational monitoring.